# ThreadPool in Python

ThreadPool allows you to execute multiple tasks concurrently using a pool of worker threads. This notebook covers theory, practical examples, and best practices.

## What is ThreadPool?

A **ThreadPool** is a collection of pre-allocated threads waiting to execute tasks. Instead of creating a new thread for each task (which is expensive), you submit tasks to the pool, and worker threads pick them up and execute them.

### Key Benefits:
- **Efficiency**: Reuses threads instead of creating/destroying them
- **Concurrency**: Execute multiple I/O-bound tasks simultaneously
- **Resource Control**: Limits the number of concurrent threads
- **Easier Management**: Handles thread lifecycle automatically

## Theory: How ThreadPool Works

```
Task Queue                Worker Threads
┌─────────┐              ┌──────────────┐
│ Task 1  │──────────────│ Thread 1     │
├─────────┤     ┌─────────────────────────┤
│ Task 2  │─┐   │        Thread 2     │
├─────────┤ │   ├──────────────────────────┤
│ Task 3  │─┼──→│        Thread 3     │
├─────────┤ │   └──────────────────────────┘
│ Task 4  │─┘
└─────────┘
```

1. Tasks are submitted to a queue
2. Idle worker threads pick up tasks from the queue
3. Thread executes the task and returns the result
4. Thread becomes available for next task
5. Unused threads sleep, conserving resources

### Use ThreadPool for:
- **I/O-bound operations**: Network requests, file operations, database queries
- **NOT for CPU-bound**: Use `multiprocessing.Pool` instead (due to GIL)

## Example 1: Basic ThreadPool with map()

In [1]:
from concurrent.futures import ThreadPoolExecutor
import time

# Simulating I/O-bound operation
def fetch_data(item):
    """Simulates fetching data with a 1-second delay"""
    time.sleep(1)
    return f"Data for {item}"

# Sequential approach (slow)
print("Sequential approach:")
start = time.time()
results_seq = [fetch_data(i) for i in range(3)]
print(f"Time: {time.time() - start:.2f}s")
print(f"Results: {results_seq}\n")

# ThreadPool approach (fast)
print("ThreadPool approach:")
start = time.time()
with ThreadPoolExecutor(max_workers=3) as executor:
    results_pool = list(executor.map(fetch_data, range(3)))
print(f"Time: {time.time() - start:.2f}s")
print(f"Results: {results_pool}")

Sequential approach:
Time: 3.00s
Results: ['Data for 0', 'Data for 1', 'Data for 2']

ThreadPool approach:
Time: 1.00s
Results: ['Data for 0', 'Data for 1', 'Data for 2']


## Example 2: Using submit() and as_completed()

In [3]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

def api_call(url, delay):
    """Simulates API call with varying delays"""
    time.sleep(delay)
    return f"Response from {url}"

# Using submit() for more control
print("Using submit() with as_completed():\n")
with ThreadPoolExecutor(max_workers=3) as executor:
    # Submit tasks and get Future objects
    futures = {
        executor.submit(api_call, f"http://api{i}.com", delay): i 
        for i, delay in enumerate([2, 1, 3], 1)
    }
    
    # Process results as they complete (not in order of submission)
    for future in as_completed(futures):
        idx = futures[future]
        result = future.result()
        print(f"Task {idx} completed: {result}")

Using submit() with as_completed():

Task 2 completed: Response from http://api2.com
Task 1 completed: Response from http://api1.com
Task 3 completed: Response from http://api3.com


## Example 3: Real-world - Parallel File Download Simulation

In [4]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

def download_file(filename):
    """Simulates downloading a file"""
    file_size = len(filename)
    download_time = file_size * 0.1  # Longer names = longer downloads
    time.sleep(download_time)
    return {"file": filename, "status": "downloaded", "size_mb": file_size}

files = ["document.pdf", "video.mp4", "archive.zip", "data.xml", "report.docx"]

# Parallel downloads
print("Downloading files in parallel...\n")
start = time.time()

with ThreadPoolExecutor(max_workers=3) as executor:
    # Submit all download tasks
    future_to_file = {
        executor.submit(download_file, f): f for f in files
    }
    
    # Process as each download completes
    for future in as_completed(future_to_file):
        result = future.result()
        elapsed = time.time() - start
        print(f"[{elapsed:.2f}s] ✓ {result['file']} ({result['size_mb']} MB)")

total_time = time.time() - start
print(f"\nCompleted in {total_time:.2f}s (vs {sum(len(f)*0.1 for f in files):.2f}s sequentially)")


[0.91s] ✓ video.mp4 (9 MB)
[1.11s] ✓ archive.zip (11 MB)
[1.21s] ✓ document.pdf (12 MB)
[1.71s] ✓ data.xml (8 MB)
[2.21s] ✓ report.docx (11 MB)

Completed in 2.21s (vs 5.10s sequentially)


## Best Practices & Key Takeaways

### ✅ Do's:
- Use ThreadPool for **I/O-bound operations** (network, files, databases)
- Set `max_workers` based on your system and workload (typically 4-8 for I/O)
- Use `with` statement for automatic thread cleanup
- Use `as_completed()` when task completion order matters
- Handle exceptions in worker functions or with `future.result()`

### ❌ Don'ts:
- Don't use ThreadPool for **CPU-bound tasks** (use `multiprocessing.Pool`)
- Don't create unlimited threads (set reasonable `max_workers`)
- Don't ignore exceptions from `future.result()`
- Don't forget to wait for all tasks (daemon threads may not complete)

### Methods Summary:
- **`map(func, iterable)`**: Process many items, returns results in order
- **`submit(func, *args)`**: Submit individual task, get Future object immediately
- **`as_completed(futures)`**: Iterate futures as they complete
- **`executor.shutdown()`**: Wait for all tasks (automatic with `with` statement)